Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# 1. Producer: stream order events into an Iceberg table

This is the first of three notebooks that run **at the same time**, each in its own tab:

| Notebook | What it does |
|---|---|
| **1. Producer** -- this one | Generates order events with Spark's `rate` source and appends them to an Iceberg table. |
| 2. Consumer | Reads that table as a stream with `format("iceberg")` and upserts every micro-batch into an ADW table. |
| 3. Source table changes | Updates, deletes, overwrites, merges, evolves and compacts the source table while 1 and 2 run, and shows what each change does to the streams. |

Start this notebook first, then notebook 2, then work through notebook 3. The one-time setup -- Volume, Credential Store entry, ADW user -- is in the README.

**Why one notebook per stream?** On AIDP, while a stream started from a notebook is running, the other cells of *that* notebook cannot run: they wait and then fail with "Command execution failed on compute cluster". Closing the notebook does not stop the stream either. So the stream cell below keeps running, and it always stops the stream before it returns: when notebook 3 asks it to, or after `RUN_MINUTES`.

## Configuration

In [ ]:
# ---- shared configuration: keep these values identical in all three notebooks ---------------
CATALOG = "default"
SCHEMA = "iceberg_streaming_demo"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.order_events"

# The Iceberg table lives in your bucket as <WAREHOUSE>/<schema>/<table>. ADW's Hadoop catalog
# support finds a table by that layout (notebook 2 creates the ADW external table).
WAREHOUSE = "oci://<bucket>@<namespace>/iceberg-streaming/warehouse"
TABLE_LOCATION = f"{WAREHOUSE}/{SCHEMA}/order_events"

# Checkpoints and stop files live on a Volume (here an external volume over a bucket folder).
STATE_DIR = "/Volumes/default/default/iceberg_checkpoints/streaming-from-iceberg-source"
CONTROL_DIR = f"{STATE_DIR}/control"

# Workspace folder holding this sample's helper modules, stream_control.py and adw_sink.py.
SAMPLE_DIR = "/Workspace/streaming-from-iceberg-source"

# ---- producer -------------------------------------------------------------------------------------
ROWS_PER_SECOND = 20      # events generated per second
NUM_ORDERS = 5000         # events cycle through this many orders: each order changes every ~4 minutes
TRIGGER = "1 minute"      # one Iceberg commit, so one snapshot, per trigger
RUN_MINUTES = 90          # the stream stops by itself after this long

assert "<" not in WAREHOUSE, "Set WAREHOUSE to your bucket and namespace."
import os
assert os.path.isdir(os.path.dirname(STATE_DIR)), f"Create the Volume first (README): {os.path.dirname(STATE_DIR)}"

In [ ]:
import sys
if SAMPLE_DIR not in sys.path:
    sys.path.insert(0, SAMPLE_DIR)
import stream_control

## Create the source table

The table is a native Iceberg table in the `default` Standard catalog, with an explicit `LOCATION` in your bucket so that ADW can read it in place (notebook 2). Its table properties matter for everything downstream:

| Property | Value | Why |
|---|---|---|
| `write.update.mode`, `write.delete.mode`, `write.merge.mode` | `copy-on-write` | Row-level changes rewrite data files instead of writing *delete files*. ADW external tables cannot read delete files -- queries fail with ORA-29913, "Delete files not supported" -- and neither can the Iceberg changelog view that notebook 3 uses to catch up. Copy-on-write is Iceberg's default; setting it explicitly keeps it from drifting. |
| `write.metadata.delete-after-commit.enabled` | `true` | A streaming writer commits every trigger, and every commit writes a new `vN.metadata.json`. With this on, Iceberg deletes the oldest ones after each commit... |
| `write.metadata.previous-versions-max` | `100` | ...keeping this many. In our test, 8 commits left 4 metadata files with a limit of 3, and 9 without cleanup. Snapshots are a separate matter: expire them as shown in notebook 3. |

| `commit.retry.num-retries` | `10` | Notebook 3 writes to the table while this stream commits. Every commit must claim the next `vN.metadata.json`; a writer that loses the race retries, 4 times by default, and then fails with `CommitFailedException: Version N already exists`. More retries give notebook 3's writes more chances; it also retries them itself. |

`PARTITIONED BY (region)` gives notebook 3 partitions to overwrite and delete.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SOURCE_TABLE} (
        event_id  BIGINT        COMMENT 'unique and increasing; orders the events of one order',
        order_id  BIGINT,
        region    STRING,
        status    STRING,
        amount    DECIMAL(12,2),
        op        STRING        COMMENT 'U = upsert the order, D = delete the order',
        event_ts  TIMESTAMP)
    USING iceberg
    PARTITIONED BY (region)
    LOCATION '{TABLE_LOCATION}'
    TBLPROPERTIES (
        'write.update.mode' = 'copy-on-write',
        'write.delete.mode' = 'copy-on-write',
        'write.merge.mode'  = 'copy-on-write',
        'commit.retry.num-retries' = '10',
        'write.metadata.delete-after-commit.enabled' = 'true',
        'write.metadata.previous-versions-max' = '100')
""")
spark.sql(f"DESCRIBE TABLE EXTENDED {SOURCE_TABLE}").where("col_name IN ('Location', 'Provider', 'Type')").show(truncate=False)

## Define the event stream

The `rate` source emits `(timestamp, value)` rows with an increasing `value`. Each row becomes one order event:

| Column | Derived from | Meaning |
|---|---|---|
| `event_id` | `value` | unique and increasing, so the newest event of an order wins downstream |
| `order_id` | `value % NUM_ORDERS` | the same orders come back again and again, so the consumer upserts |
| `region` | `order_id % 4` | an order stays in one region; also the partition |
| `status` | how many times the stream has cycled through the orders | NEW, then PAID, SHIPPED, DELIVERED |
| `op` | about 1 event in 101 | `D` asks the consumer to delete the order; every other event is `U` |

In [ ]:
from pyspark.sql import functions as F

regions = F.array(*[F.lit(r) for r in ("east", "west", "north", "south")])
statuses = F.array(*[F.lit(s) for s in ("NEW", "PAID", "SHIPPED", "DELIVERED")])
order_id = F.col("value") % NUM_ORDERS
cycle = F.floor(F.col("value") / NUM_ORDERS)          # how often every order has had an event

events = (spark.readStream.format("rate").option("rowsPerSecond", ROWS_PER_SECOND).load()
          .select(
              F.col("value").alias("event_id"),
              order_id.alias("order_id"),
              F.element_at(regions, (order_id % 4 + 1).cast("int")).alias("region"),
              F.element_at(statuses, (cycle % 4 + 1).cast("int")).alias("status"),
              (order_id % 1000 * 1.5 + cycle % 4 * 10 + 5).cast("decimal(12,2)").alias("amount"),
              F.when(F.col("value") % 101 == 100, "D").otherwise("U").alias("op"),
              F.col("timestamp").alias("event_ts")))

## Start the stream

| Writer setting | Value | Effect |
|---|---|---|
| `format("iceberg")`, `toTable(...)` | | Appends each micro-batch to the table as one Iceberg commit. The table must exist first. |
| `outputMode` | `append` | Iceberg sinks support `append` and `complete`; `complete` replaces the table on every batch. |
| `trigger(processingTime=...)` | `TRIGGER` | One commit, so one snapshot, per trigger. One minute is the minimum the Iceberg docs recommend: every commit adds a snapshot, a manifest list and small data files. It also leaves room for other writers: with a 10-second trigger, one append from notebook 3 could not commit for 16 minutes, losing every race for the next metadata version. |
| `checkpointLocation` | a Volume path | Where Spark records progress, so a restart continues where it stopped. One folder per query; never share one. |
| `fanout-enabled` | `true` | Writes rows of every partition without sorting each micro-batch by partition first; the Iceberg docs recommend it for streaming writes to partitioned tables. In our test the write also worked without it. |
| `snapshot-property.app` | a tag | Adds `app` to every commit's snapshot summary, so notebook 3 can tell the producer's commits from its own changes. |

The cell stays busy while the stream runs and prints a progress line every minute. To stop the stream, run the "Stop the streams" cell of notebook 3 -- it calls `stream_control.request_stop(CONTROL_DIR, "producer")` -- or wait for `RUN_MINUTES`.

In [ ]:
stream_control.clear_stop(CONTROL_DIR, "producer")   # forget a stop request left from an earlier run

query = (events.writeStream
         .format("iceberg")
         .outputMode("append")
         .trigger(processingTime=TRIGGER)
         .option("checkpointLocation", f"{STATE_DIR}/producer")
         .option("fanout-enabled", "true")
         .option("snapshot-property.app", "streaming-from-iceberg-source/producer")
         .queryName("iceberg_producer")
         .toTable(SOURCE_TABLE))

stream_control.run_until_stopped(query, CONTROL_DIR, "producer", max_minutes=RUN_MINUTES)

## What the stream wrote

Run this once the stream cell above has returned. Every commit of the producer is an `append` snapshot tagged with `app`. Notebook 3's changes show up as other operations, without the tag.

In [ ]:
spark.sql(f"""
    SELECT coalesce(summary['app'], '(notebook 3)') AS writer, operation, count(*) AS snapshots,
           min(committed_at) AS first_commit, max(committed_at) AS last_commit
    FROM {SOURCE_TABLE}.snapshots
    GROUP BY 1, 2 ORDER BY 4""").show(truncate=False)
spark.sql(f"SELECT count(*) AS events, count(DISTINCT order_id) AS orders FROM {SOURCE_TABLE}").show()

## Clean up -- optional

Run this after notebooks 2 and 3 are done; notebook 2 cleans up the ADW side. `DROP TABLE` -- even with `PURGE` -- removes only the catalog entry of a table with its own `LOCATION`: its files stay in the bucket. A later `CREATE TABLE` at that location would attach to the old metadata, and fails if the schemas differ ("Existing Iceberg schema at location ... does not match incoming schema"). So this cell also deletes the table's folder, `TABLE_LOCATION`, and nothing else.

In [ ]:
CLEAN_UP = False   # set to True to drop the source table, delete its files and the streaming state

def delete_folder(location, attempts=3):
    """Delete an Object Storage folder and check it is gone: one recursive delete of a large
    folder once left about half of its ~1,000 objects behind."""
    folder = spark._jvm.org.apache.hadoop.fs.Path(location)
    fs = folder.getFileSystem(spark._jsc.hadoopConfiguration())
    for _ in range(attempts):
        if not fs.exists(folder):
            return
        fs.delete(folder, True)
    if fs.exists(folder):
        raise RuntimeError(f"Could not delete {location}; delete it in the OCI Console before re-running notebook 1.")


if CLEAN_UP:
    import shutil
    spark.sql(f"DROP TABLE IF EXISTS {SOURCE_TABLE}")
    delete_folder(TABLE_LOCATION)
    shutil.rmtree(STATE_DIR, ignore_errors=True)
    print(f"Dropped {SOURCE_TABLE}, deleted {TABLE_LOCATION} and removed {STATE_DIR}")